In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\Projects\\Kidney-Disease-Classification-Deep-Learning-Project'

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list



@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [5]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [15]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)



    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        get_model = self.config.prepare_base_model
        params = self.params
        training_data = self.config.data_transformation

        create_directories([training.root_dir])

        training_config = TrainingConfig(
            root_dir=training.root_dir,
            trained_model_path=training.trained_model_path,
            updated_base_model_path=get_model.model_path,
            training_data=training_data.root_dir,
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config



    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
            config = self.config.prepare_callbacks
    
            model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
    
            create_directories([config.root_dir, config.tensorboard_root_log_dir, model_ckpt_dir])
    
            prepare_callbacks_config = PrepareCallbacksConfig(
                root_dir=config.root_dir,
                tensorboard_root_log_dir=config.tensorboard_root_log_dir,
                checkpoint_model_filepath=config.checkpoint_model_filepath
            )
    
            return prepare_callbacks_config

In [7]:
import keras
import time
from cnnClassifier import logger

In [12]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config



    def get_model(self):
        model = keras.models.load_model(self.config.updated_base_model_path)
        return model


    def train_valid_test_generator(self):
        datagen = keras.src.legacy.preprocessing.image.ImageDataGenerator(
            preprocessing_function=keras.applications.vgg16.preprocess_input
        )

        if self.config.params_is_augmentation:
            train_datagen = keras.src.legacy.preprocessing.image.ImageDataGenerator(
                preprocessing_function=keras.applications.vgg16.preprocess_input,
                rotation_range=20,
                horizontal_flip=True,
                width_shift_range=0.1,
                height_shift_range=0.1,
                zoom_range=0.1,
            )
        else:
            train_datagen = datagen

        self.train_generator = train_datagen.flow_from_directory(
            os.path.join(self.config.training_data, "train"),
            target_size=self.config.params_image_size[:2],
            batch_size=self.config.params_batch_size,
            class_mode="categorical",
            shuffle=True
        )

        self.valid_generator = datagen.flow_from_directory(
            os.path.join(self.config.training_data, "val"),
            target_size=self.config.params_image_size[:2],
            batch_size=self.config.params_batch_size,
            class_mode="categorical",
            shuffle=False
        )

        self.test_generator = datagen.flow_from_directory(
            os.path.join(self.config.training_data, "test"),
            target_size=self.config.params_image_size[:2],
            batch_size=self.config.params_batch_size,
            class_mode="categorical",
            shuffle=False
        )



    def train(self, callbacks_list: list):
        model = self.get_model()

        self.train_valid_test_generator()

        model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            validation_data=self.valid_generator,
            callbacks=callbacks_list
        )

        model.save(self.config.trained_model_path)

        



class PrepareCallbacks:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config



    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}"
        )
        return keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)



    @property
    def _create_ckpt_callbacks(self):
        return keras.callbacks.ModelCheckpoint(
            self.config.checkpoint_model_filepath,
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1
        )



    @property
    def _create_early_stopping(self):
        return keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            mode="max",
            restore_best_weights=True,
            verbose=1
        )



    @property
    def _create_reduce_lr(self):
        return keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.1,
            patience=2,
            mode="max",
            min_lr=1e-7,
            verbose=1
        )



    def get_callbacks(self):
        return [
            self._create_tb_callbacks,
            self._create_ckpt_callbacks,
            self._create_early_stopping,
            self._create_reduce_lr
        ]

In [16]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallbacks(config=prepare_callbacks_config)
    callbacks_list = prepare_callbacks.get_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.train(callbacks_list=callbacks_list)
except Exception as e:
    raise e

[2026-09-06 16:46:49,800: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-06 16:46:49,803: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-06 16:46:49,806: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-09-06 16:46:49,807: INFO: common: created directory at: artifacts/prepare_callbacks/tensorboard_log_dir]
[2026-09-06 16:46:49,808: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoint_dir]
[2026-09-06 16:46:49,810: INFO: common: created directory at: artifacts/training]


c:\Users\mahes\miniconda3\envs\kidney\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam_2', because it has 6 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


Found 8710 images belonging to 4 classes.
Found 1867 images belonging to 4 classes.
Found 1869 images belonging to 4 classes.
Epoch 1/20


KeyboardInterrupt: 